# 9. Reproducibilidad y bibliografía

## 9.1 Entorno

* Python 3.11.16 en macOS (Apple Silicon, 10 núcleos, 24 GB de RAM). El dashboard corre en Render con
  Python 3.11.16 y las versiones exactas de `requirements.txt`; el entorno completo de los experimentos
  y del libro está en `requirements-dev.txt`.
* Semilla global 42 (`src/config.py`), propagada a numpy, scikit-learn, XGBoost, imbalanced-learn,
  Optuna y DEAP. El AJ-KL es determinista (no tiene pasos aleatorios); la semilla solo fija el TPE que
  elige sus hiperparámetros y la submuestra del control de 40 mil filas.
* Los datos crudos del panel están en `data/raw/` (48 MB); las predicciones por fila de los 140 modelos
  del Entregable 2 (300 MB) no se versionan en GitHub y se regeneran con los scripts de ese entregable.

## 9.2 Orden de ejecución

| Paso | Script | Salida | Tiempo aproximado |
|---|---|---|---|
| 1 | Entregable 2: `scripts/03_preparar_pliegues.py` … `10_comparacion.py` | pliegues, tabla maestra de 140 modelos y predicciones | 29 h-CPU |
| 2 | `scripts/11_busqueda_literatura.py` y el registro de Web of Science | `data/revision/` | manual |
| 3 | `scripts/16_sota_sipsa.py <LightGBM\|CatBoost> <pliegue>` | `data/resultados/sota/` | 5-20 min por pliegue |
| 4 | `scripts/22_ajkl.py <pliegue>` (k1-k4 y final; modelo original) | `data/resultados/ajkl/` | 1-15 min por pliegue |
| 5 | `scripts/23_externos.py <conjunto>` y `scripts/23b_externos_combinador.py` | `data/resultados/externos/` | 1 h por conjunto |
| 6 | `scripts/13_artefactos.py todos`, `13b_aligerar.py` y `13c_artefacto_ajkl.py` | `artifacts/`, `artefactos_resumen.csv` | 40 min |
| 7 | `scripts/14_validacion_estadistica.py` y `14b_validacion_produccion.py` | `data/resultados/validacion/` | 15 min |
| 8 | `scripts/25_robustez_ajkl.py <pliegue> <experimento>` (k1-k4; sensibilidad, semillas, ruido, desbalance, muestra) | `data/resultados/robustez_ajkl/` | 1-40 min por combinación |
| 9 | `scripts/17_datos_dashboard.py` (EDA y modelos) y `scripts/26_mapa_dashboard.py` (mapa de corredores) | `data/dashboard/` | 15 min |
| 10 | `scripts/20_tabla_maestra.py` | `data/resultados/tabla_maestra.csv` | segundos |
| 11 | `python app.py` y `scripts/27_capturas_dashboard.py` (capturas del capítulo 7) | `book/figuras/` | 2 min |
| 12 | `python src/py2nb.py cuadernos/X.py book/X.ipynb --ejecutar` y `jupyter-book build book/` | `book/_build/html` | 5 min |

En macOS sin `brew install libomp`, XGBoost y LightGBM no encuentran OpenMP; `scripts/ejecutar_mac.sh` corre
cualquier script con el libomp que trae scikit-learn (por ejemplo, `scripts/ejecutar_mac.sh app.py`). El
entorno virtual conviene crearlo fuera de iCloud (por ejemplo, en `~/.venvs/entregable3`): dentro de una carpeta
sincronizada, la importación de las librerías puede quedarse esperando a que se descarguen los archivos.

Las pruebas unitarias se corren con `pytest tests/`: verifican las propiedades del AJ-KL (aprendizaje en
datos sintéticos con solución conocida, reproducibilidad, entradas inválidas, compatibilidad con
`sklearn.base.clone` y con los scorers de scikit-learn, óptimo del descenso espejo frente a SLSQP y L-BFGS,
casos límite, calibración beta monótona), el modelo
exploratorio EJUL, la explicación SHAP por permutaciones y la prueba de ida y vuelta de los artefactos.
Los scripts `12_modelo_original.py`, `15_ablacion.py`, `18_complejidad.py`, `19_robustez.py` y
`21_ejul_catboost.py` corresponden al modelo exploratorio EJUL.

In [1]:
import subprocess
import sys

import pandas as pd
from IPython.display import display

r = subprocess.run([sys.executable, "-m", "pytest", "../tests", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])

........................................................................ [ 86%]
...........                                                              [100%]
83 passed in 3.49s



## 9.3 Tabla maestra de experimentos

La tabla maestra reúne las 140 combinaciones del Entregable 2, el modelo original y los dos comparadores
del estado del arte, con las métricas de cada pliegue externo y de la prueba 2025, los tiempos y la
semilla. Está en `data/resultados/tabla_maestra.csv`.

In [2]:
maestra = pd.read_csv("../data/resultados/tabla_maestra.csv")
print(maestra.shape)
cols = ["tarea", "modelo", "balanceo", "optimizador", "pliegue", "auc", "auc_pr", "brier", "ece", "f1", "rmse", "t_total", "semilla"]
display(maestra[[c for c in cols if c in maestra.columns]].tail(15))

(715, 50)


,tarea,modelo,balanceo,optimizador,pliegue,auc,auc_pr,brier,ece,f1,rmse,t_total,semilla
700,clf,AJ-KL (original),Sin balanceo,Bayesiana (TPE),final,0.798477,0.553819,0.151487,0.005738,0.577521,NaN,656.586834,42
701,clf,AJ-KL (original),Sin balanceo,Bayesiana (TPE),k1,0.784350,0.577256,0.165316,0.013215,0.591442,NaN,105.803027,42
702,clf,AJ-KL (original),Sin balanceo,Bayesiana (TPE),k2,0.785506,0.568901,0.163974,0.009266,0.589730,NaN,54.484541,42
703,clf,AJ-KL (original),Sin balanceo,Bayesiana (TPE),k3,0.790306,0.563141,0.159224,0.012562,0.582567,NaN,326.809242,42
704,clf,AJ-KL (original),Sin balanceo,Bayesiana (TPE),k4,0.789449,0.547646,0.156827,0.011690,0.574493,NaN,364.065310,42
705,clf,CatBoost,Sin balanceo,Bayesiana (TPE),final,0.798195,0.553687,0.151537,0.005741,0.577653,NaN,245.160416,42
706,clf,CatBoost,Sin balanceo,Bayesiana (TPE),k1,0.783569,0.575044,0.165512,0.009794,0.591053,NaN,314.106422,42
707,clf,CatBoost,Sin balanceo,Bayesiana (TPE),k2,0.783852,0.565884,0.164466,0.008318,0.590159,NaN,408.178321,42
708,clf,CatBoost,Sin balanceo,Bayesiana (TPE),k3,0.788735,0.562600,0.159512,0.008314,0.582224,NaN,232.542444,42
709,clf,CatBoost,Sin balanceo,Bayesiana (TPE),k4,0.787747,0.545823,0.157118,0.009064,0.575838,NaN,240.880808,42


## 9.4 Bibliografía

```{bibliography}
:all:
```